# Securing the Couchbase MCP Server with Microsoft Entra ID

This tutorial walks through **two OAuth flows** for the Couchbase MCP server, using Microsoft Entra ID as the identity provider:

| Flow | Tested with | Style |
|------|-------------|-------|
| **Flow A — Token verify** | VS Code | Static, long-lived token; no browser interaction |
| **Flow B — Non-DCR** | MCP Inspector | Manual client registration; browser-based login |

## Prerequisites

* A Microsoft Entra ID tenant with **admin access**
* The Couchbase MCP server source (`mcp-server-couchbase`)
* `uv` installed, for running the Python server
* **MCP Inspector** (`npx @modelcontextprotocol/inspector`) for Flow B, and **VS Code** for Flow A
* A running Couchbase cluster (local or remote) with credentials

## Part 1: Entra Setup

### Step 1.1: Register the MCP Server application

This app registration represents the **resource server** — the Couchbase MCP server itself.

1. Go to **Entra admin center → App registrations → New registration**.
2. Name it `Couchbase MCP Server` (or anything you like — this guide uses `Couchbase MCP Test Server`).
3. Leave the redirect URI blank (resource servers don't need one), then click **Register**.

<img src="entra_screenshots/Entra_1.png" width="500">

<img src="entra_screenshots/Entra_2.png" width="500">

### Step 1.2: Set the Application ID URI

1. In the **Couchbase MCP Server** app, go to **Expose an API**.
2. Click **Add** / **Edit** next to **Application ID URI**.
3. Set it to match your MCP server's resource URL — `http://127.0.0.1:8000/mcp` — then click **Save**.

> This must **exactly** match the URL your MCP server will run at, including the `/mcp` path. The server's PRM (Protected Resource Metadata) document advertises this exact value, and Entra requires the Application ID URI to match it.

<img src="entra_screenshots/Entra_3.png" width="500">

<img src="entra_screenshots/Entra_4.png" width="500">

### Step 1.3: Define the scopes

1. Still in **Expose an API**, click **+ Add a scope** and create:
   * **Scope name:** `couchbase-mcp:read`
   * **Who can consent:** Admins and users
   * **Consent display name & description:** e.g. *"Read Couchbase MCP data"*
2. Click **Add scope**.
3. Repeat for `couchbase-mcp:write`.

<img src="entra_screenshots/Entra_6.png" width="500">

<img src="entra_screenshots/Entra_7.png" width="300">

### Step 1.4: (Optional) Modernize the token version

1. Go to **Manifest**.
2. Find `"requestedAccessTokenVersion"` and change `null` to `2`.
3. Click **Save**.

<img src="entra_screenshots/Entra_9.png" width="500">

This ensures Entra issues **v2 access tokens**, where the `aud` claim follows the Application ID URI convention rather than the raw client GUID.

> **Note:** When the client is registered as a Single-Page Application, Entra may still issue the GUID as `aud` — see the troubleshooting note in Part 3.

### Step 1.5: Register the Client application

This app registration represents the **client** — the tool connecting to the MCP server (VS Code or MCP Inspector).

1. Go to **App registrations → New registration**.
2. Name it `Couchbase MCP Client`, then click **Register**.

<img src="entra_screenshots/Entra_10.png" width="500">

### Step 1.6: Configure authentication platforms

The client needs **two platform types** registered, because MCP Inspector and VS Code use different redirect mechanisms.

**For VS Code (native/desktop flow):**

1. Click **+ Add a platform → Mobile and desktop applications**.
2. Add these redirect URIs:
   * `https://login.microsoftonline.com/common/oauth2/nativeclient`
   * `http://127.0.0.1:33418`
   * `https://vscode.dev/redirect`
3. Click **Configure**.

**For MCP Inspector (browser-based, SPA flow):**

1. Go to **Authentication → + Add a platform → Single-page application**.
2. Set the Redirect URI to `http://localhost:6274/oauth/callback`.
3. Click **Configure**.

<img src="entra_screenshots/Entra_11.png" width="500">

> ⚠️ A redirect URI value can only exist under **one** platform type at a time. If you need the same URI for both, use a slightly different one per platform (e.g. swap `localhost` for `127.0.0.1`).

### Step 1.7: Grant API permissions

First, grant the Couchbase MCP scopes:

1. Go to **API permissions → + Add a permission**.
2. Click **My APIs** (or **APIs my organization uses**) and select `Couchbase MCP Server` (named `Couchbase MCP Test Server` in this guide).
3. Choose **Delegated permissions**.
4. Check both `couchbase-mcp:read` and `couchbase-mcp:write`.
5. Click **Add permissions**.
6. Click **Grant admin consent for [tenant]** and confirm.

<img src="entra_screenshots/Entra_12.png" width="500">

<img src="entra_screenshots/Entra_13.png" width="500">

<img src="entra_screenshots/Entra_14.png" width="500">

Then add the Microsoft Graph OpenID permissions:

1. Click **Microsoft Graph → OpenID permissions** and select `email`, `offline_access`, `openid`, and `profile`.
2. Click **Add permissions** / **Update permissions**.
3. Click **Grant admin consent for [tenant]** and confirm.

<img src="entra_screenshots/Entra_15.png" width="500">

### Step 1.8: Collect your IDs

Note these down — you'll need them for every command below:

| Value | Where to find it |
|-------|------------------|
| **Directory (Tenant) ID** | Couchbase MCP Server app → Overview |
| **Server Application (Client) ID** | Couchbase MCP Server app → Overview |
| **Client Application (Client) ID** | Couchbase MCP Client app → Overview |

## Part 2: Flow A (Machine-to-Machine, static token)

This flow uses a long-lived static token pasted directly into the client config — no interactive browser login at request time. Useful for CI, automation, or when the IdP doesn't support a redirect flow for the client type in use.

### Step 2.1: Start the MCP server (no PRM needed)

```bash
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="<your-couchbase-username>" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="https://login.microsoftonline.com/<TENANT_ID>/discovery/v2.0/keys" \
  --oauth-issuer="https://login.microsoftonline.com/<TENANT_ID>/v2.0" \
  --oauth-audience="<SERVER_CLIENT_ID>"
```

Replace:

* `<TENANT_ID>` with your **Directory (tenant) ID**.
* `<SERVER_CLIENT_ID>` with the raw GUID of the **Couchbase MCP Server** app (not prefixed with `api://`).

> **Note:** There's no `--oauth-mcp-base-url` here. This puts the server in pure **token-verification mode** with no PRM endpoint, matching how a static-token client expects to operate.

### Step 2.2: Obtain a static access token (Device Code Flow)

Since this flow has no browser-redirect step at request time, the cleanest way to mint a token is the OAuth 2.0 **Device Authorization Grant** (Device Code Flow). It avoids local redirect ports entirely — the terminal gives you a short 8-character code to enter on a Microsoft page, and you get a token back directly.

> ⚠️ Before using this flow, make sure the Device Code flow is enabled on the **Couchbase MCP Client** app: go to **Authentication → Advanced settings** and enable **Allow public client flows**.

<img src="entra_screenshots/Entra_17.png" width="500">

**1. Request the verification codes.** In your VS Code terminal, run:

```bash
curl -X POST https://login.microsoftonline.com/<TENANT_ID>/oauth2/v2.0/devicecode \
  -H "Content-Type: application/x-www-form-urlencoded" \
  -d "client_id=<CLIENT_APPLICATION_ID>" \
  -d "scope=http://127.0.0.1:8000/mcp/couchbase-mcp:read http://127.0.0.1:8000/mcp/couchbase-mcp:write"
```

**2. Read the response.** Entra returns a JSON block with two codes:

```json
{
  "user_code": "BCDFGHJK",
  "device_code": "AgAAAAEAAAA...",
  "verification_uri": "https://microsoft.com/devicelogin",
  "expires_in": 900,
  "interval": 5
}
```

<img src="entra_screenshots/Entra_18.png" width="500">

**3. Authorize via browser.**

* Open a browser and go to `https://microsoft.com/devicelogin`.
* Enter the short `user_code` (e.g. `BCDFGHJK`).
* Sign in with your Entra user account and accept the permission prompt.

<img src="entra_screenshots/Entra_19.png" width="500">

**4. Exchange the device code for an access token.** Back in the terminal, use the long `device_code` from step 2:

```bash
curl -X POST https://login.microsoftonline.com/<TENANT_ID>/oauth2/v2.0/token \
  -H "Content-Type: application/x-www-form-urlencoded" \
  -d "grant_type=urn:ietf:params:oauth:grant-type:device_code" \
  -d "client_id=<CLIENT_APPLICATION_ID>" \
  -d "device_code=<PASTE_THE_LONG_DEVICE_CODE_HERE>"
```

This returns a JSON payload with your `access_token`. Copy it directly into `mcp.json` in the next step.

<img src="entra_screenshots/Entra_22.png" width="500">

### Step 2.3: Configure VS Code `mcp.json`

```jsonc
{
  "servers": {
    "couchbase": {
      "type": "http",
      "url": "http://127.0.0.1:8000/mcp",
      "headers": {
        "Authorization": "Bearer <paste your access token here>"
      }
    }
  }
}
```

### Step 2.4: Start and verify

1. In VS Code, start the `couchbase-entra-static` server from the MCP panel.
2. Confirm it shows **Running** with no auth prompt (the token is already embedded).
3. Open the MCP output channel and confirm tool-registration logs appear with no `401` errors.
4. Ask Copilot Chat (or your MCP-aware extension) to call a Couchbase tool, e.g. *"list my Couchbase buckets"*.

<img src="entra_screenshots/Entra_23.png" width="500">

> ⚠️ **Token expiry:** Entra access tokens are short-lived (typically 60–90 minutes). When one expires, you'll see `401 Unauthorized` in the server log. Repeat **Step 2.2** to get a fresh token and update `mcp.json`.

## Part 3: Flow B (Non-DCR with MCP Inspector)

This flow uses manual (pre-registered) client credentials and a full browser-based login.

### Step 3.1: Start the MCP server

```bash
uvx couchbase-mcp-server \
  --transport=http \
  --connection-string="couchbase://127.0.0.1" \
  --username="Administrator" \
  --password="<your-couchbase-password>" \
  --read-only-mode=false \
  --oauth-jwks-uri="https://login.microsoftonline.com/<TENANT_ID>/discovery/v2.0/keys" \
  --oauth-issuer="https://login.microsoftonline.com/<TENANT_ID>/v2.0" \
  --oauth-audience="<SERVER_CLIENT_ID>" \
  --oauth-mcp-base-url="http://127.0.0.1:8000"
```

Use the same `<TENANT_ID>` and `<SERVER_CLIENT_ID>` as before. The only difference from Part 2 is the added `--oauth-mcp-base-url`, which turns on the PRM endpoint that MCP Inspector relies on to discover the authorization server.

Confirm the server started and is advertising its PRM correctly:

```bash
curl http://127.0.0.1:8000/.well-known/oauth-protected-resource/mcp | python3 -m json.tool
```

You should see:

```json
{
  "resource": "http://127.0.0.1:8000/mcp",
  "authorization_servers": ["https://login.microsoftonline.com/<TENANT_ID>/v2.0"],
  "scopes_supported": ["couchbase-mcp:read", "couchbase-mcp:write"]
}
```

<img src="entra_screenshots/Entra_24.png" width="500">

### Step 3.2: Configure MCP Inspector

Launch MCP Inspector:

```bash
npx @modelcontextprotocol/inspector
```

Go to `http://localhost:6274` and fill in:

* **Transport Type:** Streamable HTTP
* **URL:** `http://127.0.0.1:8000/mcp`
* **Client ID:** the **Couchbase MCP Client** app's Application (Client) ID
* **Client Secret:** *(leave blank — public client)*
* **Redirect URL:** `http://localhost:6274/oauth/callback`
* **Scope:** `http://127.0.0.1:8000/mcp/couchbase-mcp:read http://127.0.0.1:8000/mcp/couchbase-mcp:write`

> ⚠️ The scope format must be `<Application ID URI>/<scope-name>` — not just the bare scope name. Since the Application ID URI is `http://127.0.0.1:8000/mcp`, each scope becomes `http://127.0.0.1:8000/mcp/couchbase-mcp:read`.

<img src="entra_screenshots/Entra_25.png" width="500">

<img src="entra_screenshots/Entra_26.png" width="500">

### Step 3.3: Connect

1. Click **Connect**.
2. A browser window opens with the Entra sign-in page.
3. Sign in with a user account that has consented to the app (or has had admin consent granted).

On success, Inspector shows **"Successfully authenticated with OAuth"**.

<img src="entra_screenshots/Entra_27.png" width="500">

### Step 3.4: Verify

1. Click the **Tools** tab → **List Tools** — this should show all 24 Couchbase MCP tools.
2. Run a read-only tool (e.g. `get_server_status` or `list_buckets`) — should succeed.
3. Run a write tool (e.g. an upsert) — should succeed, since the token includes both scopes.

<img src="entra_screenshots/Entra_30.png" width="500">

<img src="entra_screenshots/Entra_31.png" width="500">